# 01 — Esplorazione Keypoints CSV

Questo notebook serve per esplorare il CSV grezzo dei keypoint estratti da YOLO-Pose.
Eseguilo dopo aver ricevuto il primo CSV dal Membro A.

**Obiettivi:**
- Verificare che il formato CSV sia corretto
- Visualizzare la distribuzione dei NaN
- Plottare la velocità del polso su un clip di prova
- Confermare che lo scheletro viene disegnato correttamente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Carica la configurazione
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

print('Config caricato:', list(config.keys()))

In [ ]:
# ── Carica il CSV grezzo ──────────────────────────────────────────────
CSV_PATH = '../data/keypoints/clip1.csv'   # <-- cambia con il tuo file

df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
print(f'Colonne: {list(df.columns[:10])} ... ({len(df.columns)} totali)')
df.head(3)

In [ ]:
# ── Percentuale NaN per keypoint ─────────────────────────────────────
nan_pct = {}
for i in range(17):
    col = f'kp{i}_x'
    if col in df.columns:
        nan_pct[f'kp{i}'] = df[col].isna().mean() * 100

plt.figure(figsize=(12, 4))
plt.bar(nan_pct.keys(), nan_pct.values(), color='steelblue')
plt.axhline(20, color='red', linestyle='--', label='Soglia 20% NaN')
plt.ylabel('% NaN')
plt.title('Percentuale di frame con keypoint mancante')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Velocità del polso destro (kp10) ─────────────────────────────────
vx = df['kp10_x'].diff()
vy = df['kp10_y'].diff()
speed = np.sqrt(vx**2 + vy**2)

plt.figure(figsize=(14, 4))
plt.plot(df['time_s'], speed, color='gray', linewidth=0.8)
plt.xlabel('Tempo (s)')
plt.ylabel('Velocità (pixel/frame)')
plt.title('Velocità del polso destro — grezza')
plt.tight_layout()
plt.show()

print(f'Velocità media: {speed.mean():.4f}')
print(f'Velocità max:   {speed.max():.4f}')

In [ ]:
# ── Visualizza lo scheletro su un singolo frame ───────────────────────
import cv2
from IPython.display import Image

VIDEO_PATH = '../data/raw/clip1.mp4'  # <-- cambia con il tuo file
FRAME_TO_SHOW = 100

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_TO_SHOW)
ret, frame = cap.read()
cap.release()

if ret:
    row = df[df['frame'] == FRAME_TO_SHOW]
    if len(row) > 0:
        r = row.iloc[0]
        for i in range(17):
            x = r.get(f'kp{i}_x', np.nan)
            y = r.get(f'kp{i}_y', np.nan)
            if not np.isnan(x):
                cv2.circle(frame, (int(x), int(y)), 5, (0, 255, 0), -1)

    cv2.imwrite('/tmp/frame_test.jpg', frame)
    display(Image('/tmp/frame_test.jpg'))
else:
    print('Video non trovato — controlla il percorso.')